In [ ]:
# =====================================================================
# Advanced Scikit-Learn ML Pipeline (Unified Feature & Model Processing)
# Ready to run in Google Colab
# =====================================================================

# 1. Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# =====================================================================
# Advanced Scikit-Learn ML Pipeline (Unified Feature & Model Processing)
# Ready to run in Google Colab
# =====================================================================

# 1. Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

In [1]:
# =====================================================================
# Advanced Scikit-Learn ML Pipeline (Unified Feature & Model Processing)
# Ready to run in Google Colab
# =====================================================================

# 1. Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# 2. Generate a Mock "Raw" Dataset (Simulating imperfect real-world data)
# This data contains missing values (NaNs), numerical values, and categories.
np.random.seed(42)
data_size = 1000

data = {
    'age': np.random.choice([23, 45, 31, 54, np.nan, 60, 19], size=data_size), # Missing numbers
    'annual_income': np.random.normal(loc=65000, scale=20000, size=data_size),
    'education_level': np.random.choice(['High School', 'Bachelors', 'Masters', 'PhD', None], size=data_size), # Missing categories
    'account_status': np.random.choice(['Active', 'Silver', 'Gold'], size=data_size),
    'churned': np.random.choice([0, 1], p=[0.75, 0.25], size=data_size) # Target label
}

df = pd.DataFrame(data)
print("--- Step 2: Raw Data Preview (Notice missing values) ---")
print(df.head(4))


--- Step 2: Raw Data Preview (Notice missing values) ---
    age  annual_income education_level account_status  churned
0  19.0   84630.947535     High School         Silver        0
1  54.0   71614.011141             PhD         Active        0
2   NaN   64705.634827     High School         Active        0
3  19.0   80928.108594       Bachelors         Silver        0


In [3]:
# 3. Separate features (X) and target label (y)
X = df.drop(columns=['churned'])
y = df['churned']

# Identify feature types
numerical_features = ['age', 'annual_income']
categorical_features = ['education_level', 'account_status']

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
# 4. Construct Sub-Pipelines for Data Types
# Numerical: Fill missing with median, then normalize scale
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: Fill missing with "missing_value" label, then one-hot encode text
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [5]:
# 5. Combine Preprocessors into a Single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [6]:
# 6. Assemble the Final Master Pipeline
# This single object chains data cleaning, encoding, scaling, and the classifier together.
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])


In [7]:
# 7. Train the Entire Pipeline
print("\n--- Step 7: Training the Unified Pipeline ---")
# The raw data goes in; cleaning and fitting happen entirely behind the scenes!
full_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully.")


--- Step 7: Training the Unified Pipeline ---
Pipeline trained successfully.


In [8]:
# 8. Evaluate Using Raw Test Data
print("\n--- Step 8: Evaluating Model Performance ---")
# We pass raw data directly to .predict(). The test data is transformed safely
# using the exact rules learned from the training data (preventing data leakage).
y_pred = full_pipeline.predict(X_test)

print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



--- Step 8: Evaluating Model Performance ---
Pipeline Accuracy: 69.50%

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       162
           1       0.15      0.13      0.14        38

    accuracy                           0.69       200
   macro avg       0.48      0.48      0.48       200
weighted avg       0.68      0.69      0.69       200



In [9]:
# 9. Test Production Readiness (Inference on new, unstructured raw JSON)
print("\n--- Step 9: Live Production Prediction ---")
new_customer_data = pd.DataFrame([{
    'age': np.nan,                    # Missing age
    'annual_income': 95000.00,
    'education_level': 'Masters',
    'account_status': 'Gold'
}])

# Single line inference from raw user input
prediction = full_pipeline.predict(new_customer_data)
probability = full_pipeline.predict_proba(new_customer_data)[0][1]

status = "Will Churn" if prediction[0] == 1 else "Will Stay"
print(f"New Customer Prediction: {status} (Churn Probability: {probability * 100:.2f}%)")


--- Step 9: Live Production Prediction ---
New Customer Prediction: Will Stay (Churn Probability: 5.61%)
